In [ ]:
### EXP 1：one stage classifier 30 epochs

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import matthews_corrcoef, confusion_matrix, precision_score, recall_score, f1_score
import matplotlib.pyplot as plt
import seaborn as sns
import itertools

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = '/content/MA'
output_dir = base_path  

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}

class MelDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx]) 
        mel = torch.tensor(mel, dtype=torch.float32)
        if mel.ndim == 2:  
            mel = mel.unsqueeze(0)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7):
        super(ResNet50Classifier, self).__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.base_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
        self.base_model.fc = nn.Linear(self.base_model.fc.in_features, num_classes)

    def forward(self, x):
        return self.base_model(x)

train_set_path = os.path.join(base_path, "/content/MA/train_set.xlsx")
test_set_path = os.path.join(base_path, "/content/MA/test_set.xlsx")
train_df = pd.read_excel(train_set_path)
test_df = pd.read_excel(test_set_path)

def get_file_paths_and_labels(df, base_dir):
    files, labels = [], []
    for _, row in df.iterrows():
        relative_path = row['Full_Path']  
        pathology = row['Pathology']    
        full_path = os.path.join(base_dir, relative_path.replace('\\', '/'))  
        if os.path.exists(full_path) and pathology in classes:
            files.append(full_path)
            labels.append(classes[pathology])
        else:
            print(f"Warning: File {full_path} does not exist or invalid pathology.")
    return files, labels

train_files, train_labels = get_file_paths_and_labels(train_df, base_path)
test_files, test_labels = get_file_paths_and_labels(test_df, base_path)

test_dataset = MelDataset(test_files, test_labels)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"))
    plt.close()

def save_metrics(metrics, fold):
    metrics_df = pd.DataFrame(metrics, columns=["Epoch", "MCC"])
    metrics_df.to_csv(os.path.join(output_dir, f"validation_metrics_fold_{fold}.csv"), index=False)

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(10, 8))
    sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='Blues',
                xticklabels=classes.keys(), yticklabels=classes.keys())
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"))
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted')

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.4f},{precision:.4f},{recall:.4f},{f1:.4f}\n")

cv_results = []

best_params = None
best_model = None
best_val_mcc = -float('inf') 

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset([train_files[i] for i in train_idx], [train_labels[i] for i in train_idx])
        val_dataset = MelDataset([train_files[i] for i in val_idx], [train_labels[i] for i in val_idx])
        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        for epoch in range(30):
            model.train()
            total_train_loss = 0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, axis=1).cpu().numpy())

            train_loss = total_train_loss / len(train_loader)
            val_loss = total_val_loss / len(val_loader)
            train_losses.append(train_loss)
            val_losses.append(val_loss)
            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

        mcc = matthews_corrcoef(y_true, y_pred)
        fold_metrics.append(mcc)
        print(f"Fold {fold} | MCC: {mcc:.4f}")
        fold += 1

    avg_mcc = np.mean(fold_metrics)
    std_mcc = np.std(fold_metrics)
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)
        best_model = model

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)


print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")
print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)
final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(30):
    final_model.train()
    total_train_loss = 0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / len(final_train_loader)
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final")


In [ ]:
### EXP 1：Add gender as additional input feature to the baseline epoch 30

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    matthews_corrcoef,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from collections import Counter

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


base_path = "/content/MA"   
output_dir = base_path
os.makedirs(output_dir, exist_ok=True)

train_set_path = os.path.join(base_path, "train_set.xlsx")
test_set_path  = os.path.join(base_path, "test_set.xlsx")

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}
idx2class = {v: k for k, v in classes.items()}

def sex_to_code(s):
    s = str(s).strip().upper()
    if s == "M":
        return 0.0
    if s == "F":
        return 1.0
    return 0.5

class MelDataset(Dataset):
    def __init__(self, files, labels, sex_codes):
        self.files = files
        self.labels = labels
        self.sex_codes = sex_codes

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx]) 
        mel = torch.tensor(mel, dtype=torch.float32)


        if mel.ndim == 3:
            if mel.shape[0] == 1:
                mel = mel.squeeze(0)     
            elif mel.shape[-1] == 1:
                mel = mel.squeeze(-1)    
            else:
                raise ValueError(f"Unsupported mel shape {tuple(mel.shape)} for {self.files[idx]}")
        elif mel.ndim != 2:
            raise ValueError(f"Unsupported mel ndim={mel.ndim}, shape={tuple(mel.shape)} for {self.files[idx]}")

        sex_code = float(self.sex_codes[idx])
        sex_col = torch.full((mel.shape[0], 1), sex_code, dtype=torch.float32)
        mel = torch.cat([sex_col, mel], dim=1)  

        mel = mel.unsqueeze(0)  
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label


class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super().__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.base_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_feat = self.base_model.fc.in_features  
        self.base_model.fc = nn.Sequential(
            nn.Linear(in_feat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

train_df = pd.read_excel(train_set_path)
test_df  = pd.read_excel(test_set_path)


def resolve_path(p: str, base_dir: str) -> str:
    p = str(p).strip().replace("\\", "/")
    if p.startswith("/"):  
        return p
    if len(p) >= 2 and p[1] == ":":  
        return p
    return os.path.join(base_dir, p)  

def get_file_paths_labels_sex(df, base_dir):
    files, labels, sex_codes = [], [], []
    for _, row in df.iterrows():
        full_path = resolve_path(row["Full_Path"], base_dir)
        pathology = row["Pathology"]
        sex = row["Sex"]

        if pathology not in classes:
            continue

        if os.path.exists(full_path):
            files.append(full_path)
            labels.append(classes[pathology])
            sex_codes.append(sex_to_code(sex))
        else:
            print(f"Warning: File missing -> {full_path}")
    return files, labels, sex_codes

train_files, train_labels, train_sex = get_file_paths_labels_sex(train_df, base_path)
test_files,  test_labels,  test_sex  = get_file_paths_labels_sex(test_df,  base_path)

test_dataset = MelDataset(test_files, test_labels, test_sex)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"), dpi=200)
    plt.close()


def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=[idx2class[i] for i in range(len(classes))],
        yticklabels=[idx2class[i] for i in range(len(classes))]
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"), dpi=200)
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.6f},{precision:.6f},{recall:.6f},{f1:.6f}\n")

cv_results = []
best_params = None
best_val_mcc = -float("inf")

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset(
            [train_files[i] for i in train_idx],
            [train_labels[i] for i in train_idx],
            [train_sex[i] for i in train_idx],
        )
        val_dataset = MelDataset(
            [train_files[i] for i in val_idx],
            [train_labels[i] for i in val_idx],
            [train_sex[i] for i in val_idx],
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        best_fold_mcc = -float("inf")
        best_fold_state = None

        for epoch in range(30):
            model.train()
            total_train_loss = 0.0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0.0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())

            train_loss = total_train_loss / max(1, len(train_loader))
            val_loss = total_val_loss / max(1, len(val_loader))
            train_losses.append(train_loss)
            val_losses.append(val_loss)

            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

            val_mcc = matthews_corrcoef(y_true, y_pred) if len(set(y_true)) > 1 else 0.0
            if val_mcc > best_fold_mcc:
                best_fold_mcc = val_mcc
                best_fold_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        plot_loss(train_losses, val_losses, fold)

        fold_metrics.append(best_fold_mcc)
        print(f"Fold {fold} | MCC: {best_fold_mcc:.4f}")

        ckpt_path = os.path.join(output_dir, f"best_ckpt_fold{fold}_bs{batch_size}_lr{lr}.pt")
        torch.save(best_fold_state, ckpt_path)

        fold += 1

    avg_mcc = float(np.mean(fold_metrics))
    std_mcc = float(np.std(fold_metrics))
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)

print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")

print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels, train_sex)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)

final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(30):
    final_model.train()
    total_train_loss = 0.0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / max(1, len(final_train_loader))
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

final_ckpt = os.path.join(output_dir, f"final_model_bs{best_params[0]}_lr{best_params[1]}.pt")
torch.save(final_model.state_dict(), final_ckpt)
print(f"Saved final model to: {final_ckpt}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final_AddedSexColumn")


Using device: cuda
Testing parameter combination: batch_size=32, lr=0.001
Starting Fold 1 with batch_size=32, lr=0.001
Epoch 1: Train Loss: 0.8803, Validation Loss: 0.7962
Epoch 2: Train Loss: 0.7463, Validation Loss: 0.6739
Epoch 3: Train Loss: 0.6874, Validation Loss: 0.7012
Epoch 4: Train Loss: 0.6263, Validation Loss: 0.6512
Epoch 5: Train Loss: 0.5948, Validation Loss: 0.5274
Epoch 6: Train Loss: 0.5386, Validation Loss: 0.5217
Epoch 7: Train Loss: 0.4998, Validation Loss: 0.4947
Epoch 8: Train Loss: 0.4679, Validation Loss: 0.6868
Epoch 9: Train Loss: 0.4302, Validation Loss: 0.5956
Epoch 10: Train Loss: 0.4087, Validation Loss: 0.4784
Epoch 11: Train Loss: 0.3734, Validation Loss: 0.4114
Epoch 12: Train Loss: 0.3648, Validation Loss: 0.3513
Epoch 13: Train Loss: 0.3115, Validation Loss: 0.3907
Epoch 14: Train Loss: 0.2894, Validation Loss: 0.3905
Epoch 15: Train Loss: 0.2696, Validation Loss: 0.3187
Epoch 16: Train Loss: 0.2567, Validation Loss: 0.3976
Epoch 17: Train Loss: 0.23

In [ ]:
### EXP 1：Add gender as additional input feature to the baseline epoch 20

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    matthews_corrcoef,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from collections import Counter


random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = "/content/MA"   
output_dir = base_path
os.makedirs(output_dir, exist_ok=True)

train_set_path = os.path.join(base_path, "train_set.xlsx")
test_set_path  = os.path.join(base_path, "test_set.xlsx")


classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}
idx2class = {v: k for k, v in classes.items()}


def sex_to_code(s):
    s = str(s).strip().upper()
    if s == "M":
        return 0.0
    if s == "F":
        return 1.0
    return 0.5

class MelDataset(Dataset):
    def __init__(self, files, labels, sex_codes):
        self.files = files
        self.labels = labels
        self.sex_codes = sex_codes

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx])
        mel = torch.tensor(mel, dtype=torch.float32)

        if mel.ndim == 3:
            if mel.shape[0] == 1:
                mel = mel.squeeze(0)     
            elif mel.shape[-1] == 1:
                mel = mel.squeeze(-1)   
            else:
                raise ValueError(f"Unsupported mel shape {tuple(mel.shape)} for {self.files[idx]}")
        elif mel.ndim != 2:
            raise ValueError(f"Unsupported mel ndim={mel.ndim}, shape={tuple(mel.shape)} for {self.files[idx]}")

        sex_code = float(self.sex_codes[idx])
        sex_col = torch.full((mel.shape[0], 1), sex_code, dtype=torch.float32)
        mel = torch.cat([sex_col, mel], dim=1)  
        mel = mel.unsqueeze(0) 
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label


class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super().__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.base_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_feat = self.base_model.fc.in_features 
        self.base_model.fc = nn.Sequential(
            nn.Linear(in_feat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)


train_df = pd.read_excel(train_set_path)
test_df  = pd.read_excel(test_set_path)

def resolve_path(p: str, base_dir: str) -> str:
    p = str(p).strip().replace("\\", "/")
    if p.startswith("/"):  
        return p
    if len(p) >= 2 and p[1] == ":": 
        return p
    return os.path.join(base_dir, p)  

def get_file_paths_labels_sex(df, base_dir):
    files, labels, sex_codes = [], [], []
    for _, row in df.iterrows():
        full_path = resolve_path(row["Full_Path"], base_dir)
        pathology = row["Pathology"]
        sex = row["Sex"]

        if pathology not in classes:
            continue

        if os.path.exists(full_path):
            files.append(full_path)
            labels.append(classes[pathology])
            sex_codes.append(sex_to_code(sex))
        else:
            print(f"Warning: File missing -> {full_path}")
    return files, labels, sex_codes

train_files, train_labels, train_sex = get_file_paths_labels_sex(train_df, base_path)
test_files,  test_labels,  test_sex  = get_file_paths_labels_sex(test_df,  base_path)

test_dataset = MelDataset(test_files, test_labels, test_sex)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"), dpi=200)
    plt.close()

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=[idx2class[i] for i in range(len(classes))],
        yticklabels=[idx2class[i] for i in range(len(classes))]
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"), dpi=200)
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.6f},{precision:.6f},{recall:.6f},{f1:.6f}\n")

cv_results = []
best_params = None
best_val_mcc = -float("inf")

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset(
            [train_files[i] for i in train_idx],
            [train_labels[i] for i in train_idx],
            [train_sex[i] for i in train_idx],
        )
        val_dataset = MelDataset(
            [train_files[i] for i in val_idx],
            [train_labels[i] for i in val_idx],
            [train_sex[i] for i in val_idx],
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        best_fold_mcc = -float("inf")
        best_fold_state = None

        for epoch in range(20):
            model.train()
            total_train_loss = 0.0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0.0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())

            train_loss = total_train_loss / max(1, len(train_loader))
            val_loss = total_val_loss / max(1, len(val_loader))
            train_losses.append(train_loss)
            val_losses.append(val_loss)

            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

            val_mcc = matthews_corrcoef(y_true, y_pred) if len(set(y_true)) > 1 else 0.0
            if val_mcc > best_fold_mcc:
                best_fold_mcc = val_mcc
                best_fold_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        plot_loss(train_losses, val_losses, fold)

        fold_metrics.append(best_fold_mcc)
        print(f"Fold {fold} | MCC: {best_fold_mcc:.4f}")

        ckpt_path = os.path.join(output_dir, f"best_ckpt_fold{fold}_bs{batch_size}_lr{lr}.pt")
        torch.save(best_fold_state, ckpt_path)

        fold += 1

    avg_mcc = float(np.mean(fold_metrics))
    std_mcc = float(np.std(fold_metrics))
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)

print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")

print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels, train_sex)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)

final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(20):
    final_model.train()
    total_train_loss = 0.0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / max(1, len(final_train_loader))
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

final_ckpt = os.path.join(output_dir, f"final_model_bs{best_params[0]}_lr{best_params[1]}.pt")
torch.save(final_model.state_dict(), final_ckpt)
print(f"Saved final model to: {final_ckpt}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final_AddedSexColumn")


Using device: cuda
Testing parameter combination: batch_size=32, lr=0.001
Starting Fold 1 with batch_size=32, lr=0.001
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 232MB/s]


Epoch 1: Train Loss: 0.8759, Validation Loss: 0.7475
Epoch 2: Train Loss: 0.7520, Validation Loss: 0.7841
Epoch 3: Train Loss: 0.6990, Validation Loss: 0.6462
Epoch 4: Train Loss: 0.6450, Validation Loss: 0.5651
Epoch 5: Train Loss: 0.5904, Validation Loss: 0.5517
Epoch 6: Train Loss: 0.5416, Validation Loss: 0.6144
Epoch 7: Train Loss: 0.5104, Validation Loss: 0.5526
Epoch 8: Train Loss: 0.4764, Validation Loss: 0.5328
Epoch 9: Train Loss: 0.4286, Validation Loss: 0.4665
Epoch 10: Train Loss: 0.4199, Validation Loss: 0.4460
Epoch 11: Train Loss: 0.4026, Validation Loss: 0.5069
Epoch 12: Train Loss: 0.3745, Validation Loss: 0.6208
Epoch 13: Train Loss: 0.3414, Validation Loss: 0.5839
Epoch 14: Train Loss: 0.3166, Validation Loss: 0.3333
Epoch 15: Train Loss: 0.2872, Validation Loss: 0.3691
Epoch 16: Train Loss: 0.2855, Validation Loss: 0.3774
Epoch 17: Train Loss: 0.2392, Validation Loss: 0.3293
Epoch 18: Train Loss: 0.2388, Validation Loss: 0.4222
Epoch 19: Train Loss: 0.2318, Validat

In [ ]:
### EXP 1：Add gender as additional input feature to the baseline epoch 10

In [ ]:
import os
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models
from torchvision.models import ResNet50_Weights
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    matthews_corrcoef,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)
import matplotlib.pyplot as plt
import seaborn as sns
import itertools
from collections import Counter

random_seed = 42
torch.manual_seed(random_seed)
np.random.seed(random_seed)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

base_path = "/content/MA"  
output_dir = base_path
os.makedirs(output_dir, exist_ok=True)

train_set_path = os.path.join(base_path, "train_set.xlsx")
test_set_path  = os.path.join(base_path, "test_set.xlsx")

classes = {
    "ALS": 0,
    "Covid-19": 1,
    "Dysphonie": 2,
    "HC": 3,
    "Laryngitis": 4,
    "Parkinson": 5,
    "Rekurrensparese": 6
}
idx2class = {v: k for k, v in classes.items()}

def sex_to_code(s):
    s = str(s).strip().upper()
    if s == "M":
        return 0.0
    if s == "F":
        return 1.0
    return 0.5

class MelDataset(Dataset):
    def __init__(self, files, labels, sex_codes):
        self.files = files
        self.labels = labels
        self.sex_codes = sex_codes

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        mel = np.load(self.files[idx]) 
        mel = torch.tensor(mel, dtype=torch.float32)

        if mel.ndim == 3:
            if mel.shape[0] == 1:
                mel = mel.squeeze(0)   
            elif mel.shape[-1] == 1:
                mel = mel.squeeze(-1)   
            else:
                raise ValueError(f"Unsupported mel shape {tuple(mel.shape)} for {self.files[idx]}")
        elif mel.ndim != 2:
            raise ValueError(f"Unsupported mel ndim={mel.ndim}, shape={tuple(mel.shape)} for {self.files[idx]}")

        sex_code = float(self.sex_codes[idx])
        sex_col = torch.full((mel.shape[0], 1), sex_code, dtype=torch.float32)
        mel = torch.cat([sex_col, mel], dim=1) 
        mel = mel.unsqueeze(0) 
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        return mel, label

class ResNet50Classifier(nn.Module):
    def __init__(self, num_classes=7, dropout=0.3):
        super().__init__()
        self.base_model = models.resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
        self.base_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)

        in_feat = self.base_model.fc.in_features  
        self.base_model.fc = nn.Sequential(
            nn.Linear(in_feat, 512),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(512, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.base_model(x)

train_df = pd.read_excel(train_set_path)
test_df  = pd.read_excel(test_set_path)

def resolve_path(p: str, base_dir: str) -> str:
    p = str(p).strip().replace("\\", "/")
    if p.startswith("/"): 
        return p
    if len(p) >= 2 and p[1] == ":":  
        return p
    return os.path.join(base_dir, p)  

def get_file_paths_labels_sex(df, base_dir):
    files, labels, sex_codes = [], [], []
    for _, row in df.iterrows():
        full_path = resolve_path(row["Full_Path"], base_dir)
        pathology = row["Pathology"]
        sex = row["Sex"]

        if pathology not in classes:
            continue

        if os.path.exists(full_path):
            files.append(full_path)
            labels.append(classes[pathology])
            sex_codes.append(sex_to_code(sex))
        else:
            print(f"Warning: File missing -> {full_path}")
    return files, labels, sex_codes

train_files, train_labels, train_sex = get_file_paths_labels_sex(train_df, base_path)
test_files,  test_labels,  test_sex  = get_file_paths_labels_sex(test_df,  base_path)

test_dataset = MelDataset(test_files, test_labels, test_sex)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

def plot_loss(train_losses, val_losses, fold):
    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"Train vs Validation Loss - Fold {fold}")
    plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"loss_curve_fold_{fold}.png"), dpi=200)
    plt.close()

def evaluate_on_test_set(model, test_loader, fold):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred, labels=list(range(len(classes))))
    plt.figure(figsize=(10, 8))
    sns.heatmap(
        conf_matrix,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=[idx2class[i] for i in range(len(classes))],
        yticklabels=[idx2class[i] for i in range(len(classes))]
    )
    plt.xlabel("Predicted")
    plt.ylabel("True")
    plt.title(f"Confusion Matrix - Test Set (Fold {fold})")
    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, f"test_confusion_matrix_fold_{fold}.png"), dpi=200)
    plt.close()

    mcc = matthews_corrcoef(y_true, y_pred)
    precision = precision_score(y_true, y_pred, average='weighted', zero_division=0)
    recall = recall_score(y_true, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_true, y_pred, average='weighted', zero_division=0)

    print(f"Test Set Results - Fold {fold}: MCC: {mcc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}")

    test_results_path = os.path.join(output_dir, "test_metrics.csv")
    if not os.path.exists(test_results_path):
        with open(test_results_path, "w") as f:
            f.write("Fold,MCC,Precision,Recall,F1\n")
    with open(test_results_path, "a") as f:
        f.write(f"{fold},{mcc:.6f},{precision:.6f},{recall:.6f},{f1:.6f}\n")

cv_results = []
best_params = None
best_val_mcc = -float("inf")

for batch_size, lr in itertools.product([32, 64], [1e-3, 1e-4, 1e-5]):
    print(f"Testing parameter combination: batch_size={batch_size}, lr={lr}")

    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=random_seed)
    fold = 1
    fold_metrics = []

    for train_idx, val_idx in skf.split(train_files, train_labels):
        print(f"Starting Fold {fold} with batch_size={batch_size}, lr={lr}")

        train_dataset = MelDataset(
            [train_files[i] for i in train_idx],
            [train_labels[i] for i in train_idx],
            [train_sex[i] for i in train_idx],
        )
        val_dataset = MelDataset(
            [train_files[i] for i in val_idx],
            [train_labels[i] for i in val_idx],
            [train_sex[i] for i in val_idx],
        )

        train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
        val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

        model = ResNet50Classifier(num_classes=len(classes)).to(device)
        criterion = nn.CrossEntropyLoss()
        optimizer = torch.optim.Adam(model.parameters(), lr=lr)

        train_losses, val_losses = [], []

        best_fold_mcc = -float("inf")
        best_fold_state = None

        for epoch in range(10):
            model.train()
            total_train_loss = 0.0
            for inputs, labels in train_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()
                outputs = model(inputs)
                loss = criterion(outputs, labels)
                loss.backward()
                optimizer.step()
                total_train_loss += loss.item()

            model.eval()
            total_val_loss = 0.0
            y_true, y_pred = [], []
            with torch.no_grad():
                for inputs, labels in val_loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    outputs = model(inputs)
                    loss = criterion(outputs, labels)
                    total_val_loss += loss.item()
                    y_true.extend(labels.cpu().numpy())
                    y_pred.extend(torch.argmax(outputs, dim=1).cpu().numpy())

            train_loss = total_train_loss / max(1, len(train_loader))
            val_loss = total_val_loss / max(1, len(val_loader))
            train_losses.append(train_loss)
            val_losses.append(val_loss)

            print(f"Epoch {epoch + 1}: Train Loss: {train_loss:.4f}, Validation Loss: {val_loss:.4f}")

            val_mcc = matthews_corrcoef(y_true, y_pred) if len(set(y_true)) > 1 else 0.0
            if val_mcc > best_fold_mcc:
                best_fold_mcc = val_mcc
                best_fold_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

        plot_loss(train_losses, val_losses, fold)

        fold_metrics.append(best_fold_mcc)
        print(f"Fold {fold} | MCC: {best_fold_mcc:.4f}")

        ckpt_path = os.path.join(output_dir, f"best_ckpt_fold{fold}_bs{batch_size}_lr{lr}.pt")
        torch.save(best_fold_state, ckpt_path)

        fold += 1

    avg_mcc = float(np.mean(fold_metrics))
    std_mcc = float(np.std(fold_metrics))
    print(f"Cross-Validation Results for batch_size={batch_size}, lr={lr}: Avg MCC={avg_mcc:.4f}, Std MCC={std_mcc:.4f}")
    cv_results.append((batch_size, lr, avg_mcc, std_mcc))

    if avg_mcc > best_val_mcc:
        best_val_mcc = avg_mcc
        best_params = (batch_size, lr)

cv_results_df = pd.DataFrame(cv_results, columns=["Batch Size", "Learning Rate", "Avg MCC", "Std MCC"])
cv_results_df.to_csv(os.path.join(output_dir, "cv_results.csv"), index=False)
print(cv_results_df)

print(f"Best Parameters Found: Batch Size={best_params[0]}, Learning Rate={best_params[1]}, Avg MCC={best_val_mcc:.4f}")
print("Retraining model with best parameters on full training set...")
final_train_dataset = MelDataset(train_files, train_labels, train_sex)
final_train_loader = DataLoader(final_train_dataset, batch_size=best_params[0], shuffle=True)

final_model = ResNet50Classifier(num_classes=len(classes)).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(final_model.parameters(), lr=best_params[1])

for epoch in range(10):
    final_model.train()
    total_train_loss = 0.0
    for inputs, labels in final_train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = final_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_train_loss += loss.item()

    avg_train_loss = total_train_loss / max(1, len(final_train_loader))
    print(f"Retrain Epoch {epoch + 1}: Train Loss: {avg_train_loss:.4f}")

final_ckpt = os.path.join(output_dir, f"final_model_bs{best_params[0]}_lr{best_params[1]}.pt")
torch.save(final_model.state_dict(), final_ckpt)
print(f"Saved final model to: {final_ckpt}")

print("Evaluating final model on test set...")
evaluate_on_test_set(final_model, test_loader, "Final_AddedSexColumn")


Using device: cuda
Testing parameter combination: batch_size=32, lr=0.001
Starting Fold 1 with batch_size=32, lr=0.001
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 186MB/s]


Epoch 1: Train Loss: 0.8630, Validation Loss: 0.7900
Epoch 2: Train Loss: 0.7753, Validation Loss: 0.7598
Epoch 3: Train Loss: 0.6992, Validation Loss: 0.7591
Epoch 4: Train Loss: 0.6453, Validation Loss: 0.5701
Epoch 5: Train Loss: 0.6037, Validation Loss: 0.5492
Epoch 6: Train Loss: 0.5463, Validation Loss: 0.6337
Epoch 7: Train Loss: 0.5087, Validation Loss: 0.4540
Epoch 8: Train Loss: 0.4669, Validation Loss: 0.4590
Epoch 9: Train Loss: 0.4361, Validation Loss: 0.4333
Epoch 10: Train Loss: 0.4055, Validation Loss: 0.4418
Fold 1 | MCC: 0.6612
Starting Fold 2 with batch_size=32, lr=0.001
Epoch 1: Train Loss: 0.8704, Validation Loss: 1.0682
Epoch 2: Train Loss: 0.7446, Validation Loss: 0.7408
Epoch 3: Train Loss: 0.6634, Validation Loss: 0.7172
Epoch 4: Train Loss: 0.6084, Validation Loss: 0.5783
Epoch 5: Train Loss: 0.5594, Validation Loss: 0.5629
Epoch 6: Train Loss: 0.5247, Validation Loss: 0.5582
Epoch 7: Train Loss: 0.4876, Validation Loss: 0.4523
Epoch 8: Train Loss: 0.4438, Val